# Goal
Identify products and departments driving customer retention (reorder behavior)

import pandas as pd

orders = pd.read_csv('../data/orders.csv')

orders.head()

# 전체적인 데이터 구조 확인
orders.info()

In [12]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import sqlite3

#Connect to SQLite Database
conn = sqlite3.connect(':memory:')

#Load Datasets
orders = pd.read_csv('../data/orders.csv')
order_products = pd.read_csv('../data/order_products__prior.csv')
products = pd.read_csv('../data/products.csv')
aisles = pd.read_csv('../data/aisles.csv')
departments = pd.read_csv('../data/departments.csv')

#Write to SQL Tables
orders.to_sql('orders', conn, index=False, if_exists='replace')
order_products.to_sql('order_products', conn, index=False, if_exists='replace')
products.to_sql('products', conn, index=False, if_exists='replace')
aisles.to_sql('aisles', conn, index=False, if_exists='replace')
departments.to_sql('departments', conn, index=False, if_exists='replace')

## Data Overview

In [ ]:
#Overview of Datasets
print("Orders:", orders.shape)
print("Order Products:", order_products.shape)
print("Products:", products.shape)
print("Departments:", departments.shape)
print("Aisles:", aisles.shape)

Orders: (3421083, 7)
Order Products: (32434489, 4)
Products: (49688, 4)
Departments: (21, 2)
Aisles: (134, 2)


### Business Interpretation
- **Orders dataset:** ~3.4M orders from users
- **Order Products dataset:** ~34M order lines, showing which products were reordered.  
- **Products dataset:** ~50K products across multiple departments and aisles.  
- **Departments & Aisles:** provide categorical context for grouping analysis.

## Reorder Rate Analysis
The reorder rate measures the percentage of previously purchased items that customers buy again. 
This serves as a proxy for customer loyalty and product stickiness.

In [ ]:
#Reorder Rate
pd.read_sql_query("""
SELECT
    AVG(reordered) AS overall_reorder_rate
FROM order_products;
""", conn)

,overall_reorder_rate
0,0.589697


### Business Interpretation
The overall reorder rate of about 59% indicates that more than half of all purchased products are repeat purchases.

This suggests strong habitual buying behavior.

## Department Reorder Rate Analysis
This analysis measures reorder rates across product departments to identify
which categories drive the most repeat purchases.

By segmenting reorder behavior at the department level, we can better understand
customer loyalty patterns and determine which types of products exhibit stronger
habit-forming purchasing behavior.

In [ ]:
#Department Reorder Rate
pd.read_sql_query("""
SELECT
    d.department,
    AVG(op.reordered) AS reorder_rate,
    COUNT(*) AS total_orders
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY reorder_rate DESC;
""", conn)

,department,reorder_rate,total_orders
0,dairy eggs,0.669969,5414016
1,beverages,0.653460,2690129
2,produce,0.649913,9479291
3,bakery,0.628141,1176787
4,deli,0.607719,1051249
5,pets,0.601285,97724
6,babies,0.578971,423802
7,bulk,0.577040,34573
8,snacks,0.574180,2887550
9,alcohol,0.569924,153696


### Business Interpretation
The analysis of reorder rates by department shows that **Dairy and Eggs** leads with a 67% reorder rate. Other high-performing departments include **Beverages, Produce, Bakery, Deli, and Pets**, all with reorder rates above 60%. These results suggest that these categories represent **staple, repeat-purchase items** that drive consistent customer engagement and loyalty.

In contrast, departments such as **Personal Care (32%), Pantry, and International** have the lowest reorder rates. These lower-performing categories likely reflect **occasion-based or less essential purchases**, which are less influential on repeat shopping behavior.


## Top Reordered Products (Retention-Driven)
This analysis identifies the products with the highest reorder rates,
restricted to items with substantial order volume.

By focusing on frequently purchased products, we isolate items that
meaningfully drive repeat purchasing behavior rather than niche items
with artificially high reorder rates due to small sample sizes.

In [ ]:
#Top Reordered Products
pd.read_sql_query("""
SELECT
    p.product_name,
    COUNT(*) AS times_ordered,
    AVG(op.reordered) AS reorder_rate
FROM order_products op
JOIN products p ON op.product_id = p.product_id
GROUP BY p.product_id
HAVING COUNT(*) > 1000
ORDER BY reorder_rate DESC
LIMIT 10;
""", conn)

,product_name,times_ordered,reorder_rate
0,Half And Half Ultra Pasteurized,2921,0.861691
1,Whole Organic Omega 3 Milk,9108,0.860233
2,Organic Lactose Free Whole Milk,8477,0.859030
3,Organic Homogenized Whole Milk,3970,0.857683
4,Ultra-Purified Water,1489,0.857623
5,"Milk, Organic, Vitamin D",20198,0.854342
6,Organic Reduced Fat Milk,35663,0.850686
7,Goat Milk,5185,0.849952
8,Banana,472565,0.843501
9,Organic Whole Milk,9842,0.841191


### Business Interpretation
The top 10 reordered products all have reorder rates above 84%, highlighting their role as **key retention drivers**. The top product is **Half & Half Ultra-Pasteurized**, and 7 of the top 10 are **various types of milk**, showing that staple dairy products dominate repeat purchases.

Notably, **bananas** rank 9th in reorder rate but have the **highest total order volume** (472,565 orders), emphasizing the importance of high-demand items in driving repeat engagement.

A common theme among these top products is a focus on **health-conscious attributes**—organic, ultra-pasteurized, ultra-purified, reduced-fat, lactose-free—which suggests that **customers repeatedly choose products aligned with healthier lifestyles**.

## Customer Loyalty Analysis
This analysis evaluates customer-level loyalty by measuring
each user's total number of orders and their average reorder rate.

By restricting the analysis to users with more than five orders,
we focus on engaged customers and avoid skewing results with
one-time or low-activity shoppers.

In [ ]:
pd.read_sql_query("""
SELECT
    o.user_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    AVG(op.reordered) AS avg_reorder_rate
FROM orders o
JOIN order_products op ON o.order_id = op.order_id
GROUP BY o.user_id
HAVING total_orders > 5
ORDER BY avg_reorder_rate DESC
LIMIT 10;
""", conn)

,user_id,total_orders,avg_reorder_rate
0,99753,99,0.989529
1,82414,92,0.981308
2,107528,53,0.980769
3,17997,99,0.979310
4,5588,91,0.978857
5,170174,47,0.978723
6,3269,81,0.978182
7,12025,45,0.977778
8,91160,85,0.976415
9,184517,70,0.976190


### Business Interpretation
The analysis of top customers reveals an extremely loyal segment. Among the top 10 users:  
- 6 have placed over 80 orders, with 2 users completing 99 orders.  
- The most active customer (user ID 997853) has 99 total orders and an average reorder rate of about 99%. 
- Even the lowest-ranked top user in terms of total orders (user ID 12025) has about a 98.8% average reorder rate across multiple orders.

These results indicate a core group of **high-frequency, highly engaged customers** who consistently repurchase the same products. This segment is likely highly responsive to **loyalty programs, personalized promotions, and subscription offerings**. Maintaining and nurturing this top cohort could significantly increase **lifetime value** and retention rates.

# Business Recommendations
Based on the analysis of reorder behavior across products, departments, and customers, we recommend the following actions to drive retention and revenue:

- **Focus promotions on high-reorder departments** such as Dairy & Eggs, Beverages, Produce, Bakery, Deli, and Pets, which consistently drive repeat purchases.  
- **Target low-frequency or new users** with reorder reminders, personalized offers, or subscription suggestions to encourage habitual purchasing.  
- **Bundle high-retention products** like milk, ultra-pasteurized dairy, and bananas to reinforce customer loyalty and increase basket size.  
- **Highlight health-oriented and organic products**, as these are popular among repeat purchasers and likely strengthen customer engagement.  
- **Prioritize inventory and availability** of top reordered products and high-demand staples to prevent stockouts and maintain customer satisfaction.

## Next Steps
- Segment customers into **loyalty tiers** (high, medium, low) to tailor marketing campaigns.  
- Analyze **reorder trends over time** to see seasonal effects and predict demand.  
- Explore **cross-department bundles** or promotions with high-reorder products.  
- Monitor **low-reorder categories** for potential improvement or marketing opportunities.  
- Investigate **product attributes** (organic, reduced-fat, lactose-free) to guide inventory and promotions.

## Explore Interactive Visualizations
For an interactive exploration of the Instacart data reorder patterns, check out my Tableua Public dashboards: https://public.tableau.com/app/profile/christian.sullivan1214